# 11 — TMDB Movie Sentiment Scores (Keyword Aggregate)

Aggregates keyword-level valence scores from `10` up to **movie-level** sentiment.
Each movie gets a composite score summarising its keyword sentiment profile.

**Input**
- `data/tmdb_movie_keyword_pairs_scl_enriched.csv` — 928,781 (movie, keyword) pairs with valence

**Aggregation strategy**
- Only **narrative + scored** keywords contribute to the sentiment score
  (`is_narrative=True` and `scl_is_scored=True`)
- Weighted mean: keywords weighted by `1 / movie_count` (rare keywords are more
  discriminative; ubiquitous keywords like "friendship" carry less signal)
- Also emit unweighted mean and keyword counts for transparency

**Output columns**

| Column | Description |
|---|---|
| `tmdb_movie_id` | Movie identifier |
| `keyword_count` | Total keywords for this movie |
| `scored_keyword_count` | Narrative + scored keywords used in aggregation |
| `sentiment_mean` | Unweighted mean valence (narrative, scored) |
| `sentiment_weighted` | Frequency-weighted mean valence |
| `sentiment_positive_frac` | Fraction of scored narrative keywords that are positive |
| `sentiment_negative_frac` | Fraction that are negative |
| `sentiment_neutral_frac` | Fraction that are neutral |
| `dominant_polarity` | Most frequent polarity bucket |


In [1]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

PAIRS_FILE = Path("data/tmdb_movie_keyword_pairs_scl_enriched.csv")
OUT_FILE   = Path("data/tmdb_movie_sentiment_scores.csv")
OUTPUT_DIR = Path("output/tmdb-movie-sentiment-scores")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
pairs = pd.read_csv(PAIRS_FILE)
print(f"Pairs: {len(pairs):,}  movies: {pairs['tmdb_movie_id'].nunique():,}  keywords: {pairs['tmdb_keyword_id'].nunique():,}")
print(pairs.dtypes)

Pairs: 928,781  movies: 281,910  keywords: 59,070
tmdb_movie_id            int64
tmdb_keyword_id          int64
name                       str
keyword_type               str
is_narrative              bool
movie_count              int64
scl_valence            float64
scl_valence_source         str
scl_polarity               str
scl_is_scored             bool
scl_is_phrase_level       bool
dtype: object


In [3]:
# Subset: narrative + scored keywords only
scored = pairs[pairs["is_narrative"] & pairs["scl_is_scored"]].copy()

print(f"All pairs           : {len(pairs):,}")
print(f"Narrative + scored  : {len(scored):,}  ({len(scored)/len(pairs)*100:.1f}%)")
print(f"Movies with ≥1 scored narrative keyword: {scored['tmdb_movie_id'].nunique():,}")

All pairs           : 928,781
Narrative + scored  : 643,852  (69.3%)
Movies with ≥1 scored narrative keyword: 212,015


In [4]:
# Frequency weight: inverse movie_count (clip to avoid /0; already ≥1 in data)
scored["weight"] = 1.0 / scored["movie_count"].clip(lower=1)

def weighted_mean(grp):
    w = grp["weight"]
    v = grp["scl_valence"]
    return float((v * w).sum() / w.sum())

agg = (
    scored
    .groupby("tmdb_movie_id")
    .apply(lambda g: pd.Series({
        "scored_keyword_count": len(g),
        "sentiment_mean":       g["scl_valence"].mean(),
        "sentiment_weighted":   weighted_mean(g),
        "sentiment_positive_frac": (g["scl_polarity"] == "positive").mean(),
        "sentiment_negative_frac": (g["scl_polarity"] == "negative").mean(),
        "sentiment_neutral_frac":  (g["scl_polarity"] == "neutral").mean(),
        "dominant_polarity":       g["scl_polarity"].value_counts().idxmax(),
    }), include_groups=False)
    .reset_index()
)

# Add total keyword count (including non-scored)
total_kw = pairs.groupby("tmdb_movie_id")["tmdb_keyword_id"].count().rename("keyword_count")
agg = agg.merge(total_kw, on="tmdb_movie_id")

# Reorder columns
agg = agg[[
    "tmdb_movie_id", "keyword_count", "scored_keyword_count",
    "sentiment_mean", "sentiment_weighted",
    "sentiment_positive_frac", "sentiment_negative_frac", "sentiment_neutral_frac",
    "dominant_polarity",
]]

print(f"Movie-level rows: {len(agg):,}")
agg.describe()

Movie-level rows: 212,015


,tmdb_movie_id,keyword_count,scored_keyword_count,sentiment_mean,sentiment_weighted,sentiment_positive_frac,sentiment_negative_frac,sentiment_neutral_frac
count,2.120150e+05,212015.000000,212015.000000,212015.000000,212015.000000,212015.000000,212015.000000,212015.000000
mean,6.103965e+05,3.954786,3.036823,0.072906,0.080305,0.601223,0.375065,0.023711
std,4.799376e+05,3.649079,3.086642,0.378820,0.389548,0.394622,0.390318,0.116318
min,2.000000e+00,1.000000,1.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000
25%,1.997955e+05,2.000000,1.000000,-0.172215,-0.160000,0.250000,0.000000,0.000000
50%,4.840180e+05,3.000000,2.000000,0.104667,0.119381,0.666667,0.333333,0.000000
75%,1.006113e+06,5.000000,4.000000,0.352000,0.362000,1.000000,0.666667,0.000000
max,1.649764e+06,102.000000,74.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
print("Dominant polarity distribution:")
print(agg["dominant_polarity"].value_counts())

Dominant polarity distribution:
dominant_polarity
positive    129739
negative     78671
neutral       3605
Name: count, dtype: int64


In [6]:
fig = px.histogram(
    agg, x="sentiment_weighted", nbins=80,
    color="dominant_polarity",
    labels={"sentiment_weighted": "Weighted sentiment score", "dominant_polarity": "Dominant polarity"},
    title="Movie sentiment distribution (narrative keyword weighted mean)"
)
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.show()

In [7]:
fig2 = px.histogram(
    agg, x="scored_keyword_count", nbins=50,
    labels={"scored_keyword_count": "Scored narrative keywords per movie"},
    title="Distribution of scored keyword count per movie"
)
fig2.show()

print(f"\nMedian scored keywords per movie: {agg['scored_keyword_count'].median():.0f}")
print(f"Movies with ≥5 scored keywords  : {(agg['scored_keyword_count'] >= 5).sum():,}")


Median scored keywords per movie: 2
Movies with ≥5 scored keywords  : 37,919


In [8]:
# Most positive and most negative movies (min 5 scored keywords for reliability)
reliable = agg[agg["scored_keyword_count"] >= 5].copy()

print(f"Movies with ≥5 scored narrative keywords: {len(reliable):,}")
print()
print("Most POSITIVE (top 10):")
display(reliable.nlargest(10, "sentiment_weighted")[["tmdb_movie_id", "scored_keyword_count", "sentiment_weighted", "sentiment_positive_frac"]])

print("\nMost NEGATIVE (top 10):")
display(reliable.nsmallest(10, "sentiment_weighted")[["tmdb_movie_id", "scored_keyword_count", "sentiment_weighted", "sentiment_negative_frac"]])

Movies with ≥5 scored narrative keywords: 37,919

Most POSITIVE (top 10):


,tmdb_movie_id,scored_keyword_count,sentiment_weighted,sentiment_positive_frac
76320,330329,5,0.924859,0.800000
111555,517468,5,0.900571,1.000000
143698,818647,6,0.900286,0.833333
134507,721705,6,0.899752,1.000000
184839,1262415,6,0.899686,1.000000
210469,1630466,5,0.896733,0.800000
68089,283749,6,0.886842,0.500000
136692,743438,5,0.878970,1.000000
194584,1419578,6,0.869100,1.000000
50151,182756,5,0.867254,0.800000



Most NEGATIVE (top 10):


,tmdb_movie_id,scored_keyword_count,sentiment_weighted,sentiment_negative_frac
207965,1597017,5,-0.981605,0.800000
59306,240339,8,-0.955928,0.875000
96380,432855,5,-0.952420,1.000000
40387,120852,8,-0.939271,0.500000
22454,49361,15,-0.936043,0.666667
81908,358192,5,-0.935604,1.000000
33452,86743,9,-0.933921,0.666667
20425,44152,5,-0.930950,0.600000
3757,8440,5,-0.926453,0.800000
191913,1381036,5,-0.923880,0.800000


In [9]:
agg.to_csv(OUT_FILE, index=False)
shutil.copy(OUT_FILE, OUTPUT_DIR / OUT_FILE.name)
print(f"Saved {len(agg):,} rows → {OUT_FILE}")
print(f"Columns: {agg.columns.tolist()}")

Saved 212,015 rows → data/tmdb_movie_sentiment_scores.csv
Columns: ['tmdb_movie_id', 'keyword_count', 'scored_keyword_count', 'sentiment_mean', 'sentiment_weighted', 'sentiment_positive_frac', 'sentiment_negative_frac', 'sentiment_neutral_frac', 'dominant_polarity']


In [10]:
import os
from datetime import date

if not os.environ.get("KAGGLE_TOKEN"):
    print("Skipping upload: set KAGGLE_TOKEN env var to upload.")
else:
    import kagglehub
    user  = kagglehub.whoami(verbose=False)["username"]
    notes = f"Pipeline run {date.today()}"
    slug  = "tmdb-movie-sentiment-scores"
    print(f"Uploading {slug} ...")
    kagglehub.dataset_upload(
        handle=f"{user}/{slug}",
        local_dataset_dir=str(OUTPUT_DIR),
        version_notes=notes,
    )
    print(f"  → kaggle.com/datasets/{user}/{slug}")

Skipping upload: set KAGGLE_TOKEN env var to upload.
